# Part 1: Prepare Epi_CA9 velocity data for eight tumor samples


In [ ]:
import scvelo as scv
import pandas as pd
import scanpy as sc

adata_merge = sc.read_h5ad('../../adata_merge_raw_barcode.h5ad')
adata_anno = sc.read_h5ad('../../adata_anno_re.h5ad')
adata_epi = sc.read_h5ad('../../leiden_detailed/adata_epi_ca9.h5ad')
adata_qc = sc.read_h5ad('../../adata_qc.h5ad')
print(adata_merge)
print(adata_anno)
print(adata_epi)
print(adata_qc)

import pandas as pd
pd.set_option('display.max_rows', 20)


adata_epi.obs['sample'].value_counts()

ldata_p3_pr4pt = sc.read_loom('../../datasets/3/GSE178481_RAW/result/GSM5392419_pr4pt/velocyto/GSM5392419_pr4pt.loom')  ### t
ldata_p11_2 = sc.read_loom('../../datasets/11/result/GSM4630029/velocyto/GSM4630029.loom')  ### t
ldata_p12_2 = sc.read_loom('../../datasets/12/GSE171306/result/GSM5222645/velocyto/GSM5222645.loom')  ## t
ldata_p14_2t = sc.read_loom('../../datasets/14/GSE156632/results/GSM4735366_2t/velocyto/GSM4735366_2t.loom')  ## t
ldata_p14_3t = sc.read_loom('../../datasets/14/GSE156632/results/GSM4735368_3t/velocyto/GSM4735368_3t.loom')  ### t
ldata_p14_5t = sc.read_loom('../../datasets/14/GSE156632/results/GSM4735372_5t/velocyto/GSM4735372_5t.loom')  ### t
ldata_p14_6t = sc.read_loom('../../datasets/14/GSE156632/results/GSM4735374_6t/velocyto/GSM4735374_6t.loom')  ### t
ldata_p14_7t = sc.read_loom('../../datasets/14/GSE156632/results/GSM4735375_7t/velocyto/GSM4735375_7t.loom')  ### t

suffixes = [
    "p3_pr4pt",
    "p11_2",
    "p12_2",
    "p14_2t",
    "p14_3t",
    "p14_5t",
    "p14_6t",
    "p14_7t",
]
for suffix in suffixes:
    globals()[f"adata_merge_{suffix}"] = adata_merge[adata_merge.obs["sample"] == suffix].copy()
    globals()[f"adata_epi_{suffix}"] = adata_epi[adata_epi.obs["sample"] == suffix].copy()
    globals()[f"adata_qc_{suffix}"] = adata_qc[adata_qc.obs["sample"] == suffix].copy()

ldata_names = [
    "ldata_p3_pr4pt",
    "ldata_p11_2",
    "ldata_p12_2",
    "ldata_p14_2t",
    "ldata_p14_3t",
    "ldata_p14_5t",
    "ldata_p14_6t",
    "ldata_p14_7t",
]

for name in ldata_names:
    print(f"\n===== {name} =====")
    print(globals()[name].obs.head())

for name in ldata_names:
    ad = globals()[name]
    # Keep the original index for traceability.
    ad.obs["original_barcode"] = ad.obs_names.copy()

    # Remove the prefix through the first colon and replace a trailing x with -1.
    ad.obs_names = (
        ad.obs_names.str.replace(r"^[^:]*:", "", regex=True)
        .str.replace(r"x$", "-1", regex=True)
    )
    print(name, ad.obs_names[:5].tolist())

for name, ad in list(globals().items()):
    if name.startswith("adata_merge_"):
        suffix = name.removeprefix("adata_merge_")
        globals()[f"raw_barcode_to_obs_{suffix}"] = dict(zip(ad.obs["raw_barcode"].tolist(), ad.obs_names.tolist()))

for name in ldata_names:
    suffix = name.removeprefix("ldata_")
    ad = globals()[name]
    mapping = globals()[f"raw_barcode_to_obs_{suffix}"]
    ad.obs["mapped_index"] = ad.obs_names.to_series().map(mapping).values

for name in ldata_names:
    suffix = name.removeprefix("ldata_")
    ad = globals()[name]
    mapping = globals()[f"raw_barcode_to_obs_{suffix}"]
    ad.obs["mapped_index"] = ad.obs_names.to_series().map(mapping).values
    print(name, "matched =", ad.obs["mapped_index"].notna().sum(), "total =", ad.n_obs)

for name in ldata_names:
    print(name)

for name in ldata_names:
    print(name, globals().get(name))

for name in ldata_names:
    ad = globals()[name]
    ad = ad[ad.obs["mapped_index"].notna()].copy()
    ad.obs_names = ad.obs["mapped_index"].tolist()
    globals()[name] = ad

suffixes = [
    "p3_pr4pt",
    "p11_2",
    "p12_2",
    "p14_2t",
    "p14_3t",
    "p14_5t",
    "p14_6t",
    "p14_7t",
]
for suffix in suffixes:
    epi = globals()[f"adata_epi_{suffix}"]
    qc = globals()[f"adata_qc_{suffix}"]
    ld = globals()[f"ldata_{suffix}"]

    keep_qc = epi.obs_names[epi.obs_names.isin(qc.obs_names)]
    keep_ld = epi.obs_names[epi.obs_names.isin(ld.obs_names)]

    globals()[f"adata_qc_{suffix}"] = qc[keep_qc].copy()
    globals()[f"ldata_{suffix}"] = ld[keep_ld].copy()

for suffix in suffixes:
    qc = globals()[f"adata_qc_{suffix}"]
    ld = globals()[f"ldata_{suffix}"]

    globals()[f"merged_{suffix}"] = scv.utils.merge(qc, ld)

for suffix in suffixes:
    merged = globals()[f"merged_{suffix}"]
    epi = globals()[f"adata_epi_{suffix}"]

    globals()[f"adata_epi_{suffix}"] = epi[merged.obs_names].copy()

for suffix in suffixes:
    merged = globals()[f"merged_{suffix}"]
    epi = globals()[f"adata_epi_{suffix}"]

    merged.obsm["X_pca"] = epi.obsm["X_pca"].copy()
    merged.obsm["X_pca_inte"] = epi.obsm["X_pca_inte"].copy()
    merged.obsm["X_umap"] = epi.obsm["X_umap"].copy()
    merged.obs["cell_subtype"] = epi.obs["cell_subtype"].copy()

from pathlib import Path
out_dir = Path(".")
out_dir.mkdir(parents=True, exist_ok=True)

for suffix in suffixes:
    adata = globals()[f"merged_{suffix}"]
    adata.write_h5ad(out_dir / f"ladata_{suffix}_epica9.h5ad")
    print(f"saved: ladata_{suffix}.h5ad")


# Part 2: Run merged-tumor VELOVI and CellRank analysis


# Epi_CA9 VELOVI + CellRank: merged tumor samples


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import torch
from velovi import preprocess_data, VELOVI

torch.set_float32_matmul_precision("high")


## Load eight tumor samples


In [ ]:
ladata_p11_2 = sc.read_h5ad("ladata_p11_2_epica9.h5ad")
ladata_p12_2 = sc.read_h5ad("ladata_p12_2_epica9.h5ad")
ladata_p14_2t = sc.read_h5ad("ladata_p14_2t_epica9.h5ad")
ladata_p14_3t = sc.read_h5ad("ladata_p14_3t_epica9.h5ad")
ladata_p14_5t = sc.read_h5ad("ladata_p14_5t_epica9.h5ad")
ladata_p14_6t = sc.read_h5ad("ladata_p14_6t_epica9.h5ad")
ladata_p14_7t = sc.read_h5ad("ladata_p14_7t_epica9.h5ad")
ladata_p3_pr4pt = sc.read_h5ad("ladata_p3_pr4pt_epica9.h5ad")

ladata_dict = {
    "p11_2": ladata_p11_2,
    "p12_2": ladata_p12_2,
    "p14_2t": ladata_p14_2t,
    "p14_3t": ladata_p14_3t,
    "p14_5t": ladata_p14_5t,
    "p14_6t": ladata_p14_6t,
    "p14_7t": ladata_p14_7t,
    "p3_pr4pt": ladata_p3_pr4pt,
}

suffixes = [
    "p11_2", "p12_2", "p14_2t", "p14_3t",
    "p14_5t", "p14_6t", "p14_7t", "p3_pr4pt",
]
no_n_suffixes = list(suffixes)

print("Tumor samples:", no_n_suffixes)


## Analysis helpers


In [ ]:
from __future__ import annotations

import inspect
import warnings
from dataclasses import dataclass
from pathlib import Path

import anndata as ad
import cellrank as cr
import scipy.sparse as sp

warnings.filterwarnings(
    "ignore",
    message=r"Moving element from .uns\['neighbors'\]\['distances'\] to .obsp\['distances'\]",
)
warnings.filterwarnings(
    "ignore",
    message=r"Moving element from .uns\['neighbors'\]\['connectivities'\] to .obsp\['connectivities'\]",
)


def patch_cellrank_numpy_compat() -> None:
    parameters = inspect.signature(np.testing.assert_array_equal).parameters
    if "x" in parameters and "y" in parameters:
        return

    original = np.testing.assert_array_equal

    def _patched_assert_array_equal(*args, **kwargs):
        if "x" in kwargs:
            kwargs["actual"] = kwargs.pop("x")
        if "y" in kwargs:
            kwargs["desired"] = kwargs.pop("y")
        return original(*args, **kwargs)

    np.testing.assert_array_equal = _patched_assert_array_equal


@dataclass
class CellRankResult:
    name: str
    adata: object
    matrix: pd.DataFrame
    counts: pd.Series
    present: pd.Series
    positions: pd.DataFrame


patch_cellrank_numpy_compat()
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


def aggregate_transition_matrix(
    transition_matrix: sp.spmatrix,
    labels: pd.Series,
    categories: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    categorical = pd.Categorical(labels.astype(str), categories=categories)
    codes = categorical.codes
    valid = codes >= 0

    row = np.flatnonzero(valid)
    col = codes[valid]
    data = np.ones(valid.sum(), dtype=np.float64)
    membership = sp.csr_matrix((data, (row, col)), shape=(len(labels), len(categories)))

    group_mass = membership.T @ transition_matrix @ membership
    group_sizes = np.asarray(membership.sum(axis=0)).ravel()
    group_mean = sp.diags(1.0 / np.maximum(group_sizes, 1.0)) @ group_mass

    matrix = pd.DataFrame(
        group_mean.toarray(),
        index=categories,
        columns=categories,
        dtype=float,
    )
    counts = pd.Series(group_sizes, index=categories, dtype=float)
    return matrix, counts


def get_group_positions(
    adata,
    group_key: str,
    categories: list[str],
    basis_preference=("X_umap_paga", "X_umap", "X_umap_leiden", "X_diffmap_paga"),
) -> pd.DataFrame:
    for basis in basis_preference:
        if basis in adata.obsm:
            coords = pd.DataFrame(
                adata.obsm[basis][:, :2],
                index=adata.obs_names,
                columns=["x", "y"],
            )
            groups = adata.obs[group_key].astype(str)
            centroids = coords.groupby(groups).mean().reindex(categories)
            if not centroids.isna().all().all():
                return centroids
    return pd.DataFrame(index=categories, columns=["x", "y"], dtype=float)


def preprocess_for_velovi(src):
    adata = src.copy()
    scv.pp.filter_genes(adata, min_shared_counts=20)
    scv.pp.normalize_per_cell(adata)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable].copy()
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    adata = preprocess_data(adata)
    return adata


def add_velovi_outputs_to_adata(adata, vae):
    latent_time = vae.get_latent_time(n_samples=25)
    velocities = vae.get_velocity(n_samples=25, velo_statistic="mean")

    scaling = (20 / latent_time.max(0)).to_numpy()

    adata.layers["velocity"] = np.asarray(velocities) / scaling[np.newaxis, :]
    adata.layers["latent_time_velovi"] = latent_time.to_numpy()

    rates = vae.get_rates()
    adata.var["fit_alpha"] = np.asarray(rates["alpha"]) / scaling
    adata.var["fit_beta"] = np.asarray(rates["beta"]) / scaling
    adata.var["fit_gamma"] = np.asarray(rates["gamma"]) / scaling
    adata.var["fit_t_"] = (
        torch.nn.functional.softplus(vae.module.switch_time_unconstr)
        .detach()
        .cpu()
        .numpy()
    ) * scaling
    adata.layers["fit_t"] = latent_time.to_numpy() * scaling[np.newaxis, :]
    adata.var["fit_scaling"] = 1.0


def run_velovi(adata, max_epochs=None):
    VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
    vae = VELOVI(adata)
    vae.to_device(DEVICE)
    vae.train(max_epochs=max_epochs)
    vae.to_device(DEVICE)
    add_velovi_outputs_to_adata(adata, vae)
    scv.tl.velocity_graph(adata)
    scv.tl.velocity_pseudotime(adata)
    return adata, vae


def compute_cellrank_group_matrix(
    adata,
    group_key="cell_subtype",
    velocity_weight=1.0,
    pseudotime_weight=0.0,
    softmax_scale=4.0,
):
    velocity_kernel = cr.kernels.VelocityKernel(
        adata,
        attr="layers",
        xkey="Ms",
        vkey="velocity",
    ).compute_transition_matrix(
        model="deterministic",
        softmax_scale=softmax_scale,
        n_jobs=1,
        show_progress_bar=False,
    )

    total_weight = velocity_weight
    kernel = velocity_kernel
    if pseudotime_weight > 0 and "velocity_pseudotime" in adata.obs:
        pseudotime_kernel = cr.kernels.PseudotimeKernel(
            adata,
            time_key="velocity_pseudotime",
        ).compute_transition_matrix(
            threshold_scheme="soft",
            n_jobs=1,
            backend="threading",
            show_progress_bar=False,
        )
        kernel = velocity_weight * velocity_kernel + pseudotime_weight * pseudotime_kernel
        total_weight += pseudotime_weight

    if total_weight != 1.0:
        kernel = (1.0 / total_weight) * kernel

    categories = sorted(pd.Series(adata.obs[group_key]).dropna().astype(str).unique().tolist())
    matrix, counts = aggregate_transition_matrix(kernel.transition_matrix, adata.obs[group_key], categories)
    present = counts.gt(0)
    positions = get_group_positions(adata, group_key=group_key, categories=categories)
    return matrix, counts, present, positions


def merge_ladatas(sample_names):
    combo = ad.concat(
        [ladata_dict[s].copy() for s in sample_names],
        join="inner",
        merge="same",
        label="sample",
        keys=list(sample_names),
        index_unique="-",
    )
    combo.obs["sample"] = combo.obs["sample"].astype(str)
    return combo


def save_result_bundle(result: CellRankResult, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    result.adata.write_h5ad(out_dir / f"{result.name}.h5ad")
    result.matrix.to_csv(out_dir / f"{result.name}_cellrank_matrix.csv")
    pd.DataFrame(
        {"count": result.counts, "present": result.present.astype(int)}
    ).to_csv(out_dir / f"{result.name}_counts.csv")


## Parameters


In [ ]:
output_root = Path("./velovi_cellrank_outputs_epi_ca9")
output_root.mkdir(parents=True, exist_ok=True)

velovi_max_epochs = None
velocity_weight = 1.0
pseudotime_weight = 0.0
softmax_scale = 4.0


## merged_no_n: merge tumor samples and run VELOVI + CellRank


In [ ]:
no_n_adata = merge_ladatas(no_n_suffixes)
no_n_adata = preprocess_for_velovi(no_n_adata)
no_n_adata, no_n_vae = run_velovi(
    no_n_adata,
    max_epochs=velovi_max_epochs,
)
no_n_matrix, no_n_counts, no_n_present, no_n_positions = compute_cellrank_group_matrix(
    no_n_adata,
    group_key="cell_subtype",
    velocity_weight=velocity_weight,
    pseudotime_weight=pseudotime_weight,
    softmax_scale=softmax_scale,
)
result_no_n = CellRankResult(
    name="ladata_no_n_8_samples",
    adata=no_n_adata,
    matrix=no_n_matrix,
    counts=no_n_counts,
    present=no_n_present,
    positions=no_n_positions,
)
save_result_bundle(result_no_n, output_root / "merged_no_n")
result_no_n.matrix


# Part 3: Draw the single-direction CellRank graph


In [ ]:
import cellrank as cr
import pandas as pd
import numpy as np
import scipy.sparse as sp
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc

adata = sc.read_h5ad("./velovi_cellrank_outputs_epi_ca9/merged_no_n/ladata_no_n_8_samples.h5ad")
group_key = "cell_subtype"

velocity_kernel = cr.kernels.VelocityKernel(
    adata,
    attr="layers",
    xkey="Ms",
    vkey="velocity",
).compute_transition_matrix(
    model="deterministic",
    softmax_scale=4.0,
    n_jobs=1,
    show_progress_bar=False,
)

categories = sorted(adata.obs[group_key].astype(str).unique().tolist())

labels = adata.obs[group_key].astype(str)
categorical = pd.Categorical(labels, categories=categories)
codes = categorical.codes
valid = codes >= 0

row = np.flatnonzero(valid)
col = codes[valid]
data = np.ones(valid.sum(), dtype=np.float64)
membership = sp.csr_matrix((data, (row, col)), shape=(len(labels), len(categories)))

group_mass = membership.T @ velocity_kernel.transition_matrix @ membership
group_sizes = np.asarray(membership.sum(axis=0)).ravel()
group_mean = sp.diags(1.0 / np.maximum(group_sizes, 1.0)) @ group_mass

matrix = pd.DataFrame(
    group_mean.toarray(),
    index=categories,
    columns=categories,
    dtype=float,
)

In [ ]:
groups = matrix.index.tolist()

coords = pd.DataFrame(
    adata.obsm["X_umap"][:, :2],
    index=adata.obs_names,
    columns=["x", "y"],
)
positions = coords.groupby(adata.obs["cell_subtype"].astype(str)).mean().reindex(groups)
counts = adata.obs["cell_subtype"].astype(str).value_counts().reindex(groups, fill_value=0)

graph = nx.DiGraph()
for g in groups:
    graph.add_node(g, size=float(counts[g]))

threshold = 0.02

for i, src in enumerate(groups):
    for dst in groups[i + 1:]:
        w_src_dst = float(matrix.at[src, dst])
        w_dst_src = float(matrix.at[dst, src])

        if max(w_src_dst, w_dst_src) <= threshold:
            continue

        if w_src_dst > w_dst_src:
            graph.add_edge(src, dst, weight=w_src_dst)
        elif w_dst_src > w_src_dst:
            graph.add_edge(dst, src, weight=w_dst_src)

pos = {g: positions.loc[g].to_numpy() for g in groups}

node_sizes = [10 + 0.2 * float(graph.nodes[g]["size"]) for g in graph.nodes]
edge_widths = [1.5 + 10.0 * graph[u][v]["weight"] for u, v in graph.edges]
edge_colors = [graph[u][v]["weight"] for u, v in graph.edges]

fig, ax = plt.subplots(figsize=(10, 8))

nx.draw_networkx_edges(
    graph,
    pos,
    edge_color=edge_colors,
    edge_cmap=plt.cm.magma,
    width=edge_widths,
    arrows=True,
    arrowsize=24,
    alpha=0.9,
    ax=ax,
    min_source_margin=12,
    min_target_margin=12,
)

nx.draw_networkx_nodes(
    graph,
    pos,
    node_size=node_sizes,
    node_color="#d9efe8",
    edgecolors="#1f3b4d",
    ax=ax,
)

nx.draw_networkx_labels(
    graph,
    pos,
    font_size=10,
    ax=ax,
)

if edge_colors:
    vmin = min(edge_colors)
    vmax = max(edge_colors)
    if vmin == vmax:
        vmax = vmin + 1e-6

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(cmap=plt.cm.magma, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("Edge weight")

ax.set_axis_off()
plt.savefig('ladata_tumor_samples_cellrank_graph_single.pdf')
plt.show()